## **Mini Project**

### **Airline Tweet Sentiment Classifier using Natural Language Processing**


**Notes:**

Use sample dataset from - https://github.com/salman1256/aimltraining/blob/main/Day-30/airline_tweets_sample.csv


**Steps:**

1. Import libraries
2. Load and explore dataset
3. Clean and preprocess the text
4. Convert text to numerical vectors (TF-IDF)
5. Split into train and test sets
6. Train a Logistic Regression model
7. Evaluate accurancy and classification report
8. Predict sentiment for new example tweets

In [15]:
#Step 1 a: Import Required Libraries

import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer

In [3]:
#Step 1 a: Import Required Libraries

nltk.download('wordnet')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
# Step 2a: Load DataSet

df = pd.read_csv('airline_tweets_sample.csv')

In [6]:
print("Sample Data Airlines Tweet", df.shape)
df.head()

Sample Data Airlines Tweet (30, 2)


,text,sentiment
0,"@United flight was delayed for 3 hours, worst ...",negative
1,"Loved the service on @Delta, crew was super fr...",positive
2,"@AmericanAir lost my luggage again, so disappo...",negative
3,Smooth boarding and on-time arrival. Great job...,positive
4,The seats were uncomfortable but staff was polite,neutral


In [13]:
# Step 3: Text Cleaning and Preprocessing
# For each tweet do:
# Convert to lowercase
# Removing URLs
# Remove special characters and numbers
# Remove stopwords (common words like *the,is,and* etc.)
# Apply **stemming** (reduce words to their root form)

def clean_text(text):
  text=str(text)
  text=re.sub(r'http\S+','',text)
  text=text.translate(str.maketrans('','',string.punctuation))
  text=re.sub(r'\s+',' ',text).strip()
  text=re.sub(r'[^\x00-\x7F]','',text)
  text=text.lower()
  return text

df['cleaned_text']=df['text'].apply(clean_text)
df['cleaned_text']

,cleaned_text
0,united flight was delayed for 3 hours worst ex...
1,loved the service on delta crew was super frie...
2,americanair lost my luggage again so disappointed
3,smooth boarding and ontime arrival great job s...
4,the seats were uncomfortable but staff was polite
5,jetblue flight attendants were rude not flying...
6,got a free upgrade to business class thank you...
7,average flight nothing special to mention
8,deltaairlines provided excellent support with ...
9,the inflight entertainment was not working


In [16]:
#Step 4:
# a) Convert text to numerical vectors (TF-IDF)
# b) check X,y and shape len

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['cleaned_text']).toarray()
y = df['sentiment']

print(f"X (Feature Matrix) Shape: {X.shape}")
print(f"y (Target Vector) Shape: {y.shape}")
print(f"Number of unique features (words) in vectorizer: {len(vectorizer.get_feature_names_out())}")

X (Feature Matrix) Shape: (30, 135)
y (Target Vector) Shape: (30,)
Number of unique features (words) in vectorizer: 135


In [17]:
#Step 5: Split into train and test sets
#80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (24, 135)
X_test shape: (6, 135)
y_train shape: (24,)
y_test shape: (6,)


In [21]:
#Step 6: Train a Logistic Regression Model
#a: Create Logistic Model
#b: Train Logistic Model

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, random_state=42) # Increased max_iter for convergence
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [22]:
# Step 7:
# Evaluate accuracy and classification report
# a. predict Model
# b. Precision, recall, F1-Score for each sentiments

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy Score: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy Score: 0.3333

Classification Report:
              precision    recall  f1-score   support

    negative       0.20      1.00      0.33         1
     neutral       0.00      0.00      0.00         1
    positive       1.00      0.25      0.40         4

    accuracy                           0.33         6
   macro avg       0.40      0.42      0.24         6
weighted avg       0.70      0.33      0.32         6



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [23]:
#Step 8: Predict sentiment for new example tweets
new_tweets = [
    "The food was amazing, seat was comfortable and friendly cabin crew!",
    "The flight delayed for 6 hours! Bad experience!",
    "Very comfortable journey, I like the on-board entertainment movies and enjoy throughout the journey",
    "Luggage missing upon touch down. Lousy airline!"
]
clean_new_tweets = [clean_text(tweet) for tweet in new_tweets]
X_new = vectorizer.transform(clean_new_tweets)
new_predictions = model.predict(X_new)
print("Sentiment Predictions for New Tweets:")
for i, tweet in enumerate(new_tweets):
    print(f"Tweet: '{tweet}' \nPredicted Sentiment: {new_predictions[i]}\n")

Sentiment Predictions for New Tweets:
Tweet: 'The food was amazing, seat was comfortable and friendly cabin crew!' 
Predicted Sentiment: positive

Tweet: 'The flight delayed for 6 hours! Bad experience!' 
Predicted Sentiment: negative

Tweet: 'Very comfortable journey, I like the on-board entertainment movies and enjoy throughout the journey' 
Predicted Sentiment: positive

Tweet: 'Luggage missing upon touch down. Lousy airline!' 
Predicted Sentiment: negative

